# Part 3: a pre-registered test on 120 MMLU questions
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/advaith334/nst2046-memory-sycophancy/blob/main/notebooks/04_preregistered_mmlu.ipynb)

Parts 1 and 2 rest on questions drafted with AI help and were designed after we saw results. So before running any trial we fixed our hypotheses, on 120 questions drawn at random from MMLU (Hendrycks et al., 2020): 60 school (biology, chemistry, mathematics) and 60 health (clinical knowledge). Run 5 November 2026.

The memory sentence here repeats the question, e.g. *"User believes that the answer to '...' is X"*, which may strengthen the effect.

The pre-registered rule compares item means across conditions, using the score the harness logged at run time.

In [ ]:
# Setup. In Colab this downloads the repo. On your own machine, start Jupyter from the repo folder.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("memsyco_lite"):
    !git clone -q https://github.com/advaith334/nst2046-memory-sycophancy
    %cd nst2046-memory-sycophancy
    !pip install --quiet openai
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from memsyco_lite import load_items, load_run, load_prompts, load_table, model_name
from memsyco_lite.data import show_prompt
from memsyco_lite.stats import effect, fmt, rate
pd.set_option("display.max_colwidth", 140)
plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
items = load_items("mmlu")
df = load_run("4-mmlu", "default", scoring="harness_score")
prompts = load_prompts("4-mmlu", "default")
r = df[(df.model == "mistral-nemo") & (df.condition == "stored_incorrect")].iloc[0]
print(show_prompt(prompts[r.prompt_id]))

## The hypotheses

In [ ]:
nemo, small = df[df.model == "mistral-nemo"], df[df.model == "mistral-small"]
E = lambda d, c, ref, v="false": fmt(effect(d, c, ref, v, paired=False), halves="up")
rows = [
    ("H1 Harm above zero, Nemo", E(nemo, "stored_incorrect", "no_belief")),
    ("H2 Harm above zero, Mistral Small", E(small, "stored_incorrect", "no_belief")),
    ("H3 Label reduces Nemo's harm", E(nemo, "stored_incorrect", "stored_incorrect_labelled")),
    ("H4 Labelled memory stays above no belief, Nemo", E(nemo, "stored_incorrect_labelled", "no_belief")),
    ("H5 Benefit above zero, Nemo", E(nemo, "stored_correct", "no_belief", "correct")),
] + [(f"H6 Harm, {model_name(m)}", E(df[df.model == m], "stored_incorrect", "no_belief")) for m in ["llama-8b-deepinfra", "qwen-small", "gemma-moe"]]
pd.DataFrame(rows, columns=["test", "result (points, 95% interval)"])

The full table as reported, with Holm-adjusted p-values and the frame tests (H7, H8, Q1, Q2):

In [ ]:
load_table("4_preregistered_tests")

## False and correct answers by model

In [ ]:
t = df.groupby(["model", "condition"])[["false", "correct"]].mean().mul(100).round(0).unstack()
t.index = [model_name(m) for m in t.index]
t

## Truth-sensitivity
Following a true memory minus following a false one, on items a model got right with no memory (it *knew*) and on items it got wrong (*unsure*). Exploratory.

In [ ]:
load_table("4_truth_sensitivity")

## What this shows
- All five primary hypotheses passed: Nemo's harm was +55 [+47, +63], and every open model moved, including Llama (+30).
- The label is weaker here, removing 28% of Nemo's harm against 56% in the main study.
- Where a model knew the answer it followed a true note far more than a false one (+37 in Nemo). Where it did not, the gap almost vanished (+6). That is the evidence for our claim.

**Limits.** The memory sentence nearly hands over the answer, and nobody has hand-checked the MMLU keys. Claude stopped at its budget cap, with only 60 items in both arms.

## Replay or live
- **Replay** (the default) uses the trials we recorded, so it needs no key and costs nothing.
- **Live** sends a small sample of the *same* prompts to the *same* models and hosts on OpenRouter, with your own key, and puts your numbers beside ours.

Put your key in Colab's 🔑 **Secrets** panel as `OPENROUTER_API_KEY`, or paste it when asked. Models answer at temperature 0.7, so your numbers will differ from ours within noise, and hosts can change or disappear over time.

In [ ]:
MODE = "replay"   # "replay": our recorded results, no key and no cost. "live": rerun a small sample with your own key.
CAP_USD = 0.50    # a live run stops once it has spent this much (US$)

In [ ]:
if MODE == "live":
    from memsyco_lite.live import Client, get_key, sample, rerun_mcq, estimate_cost, compare
    recorded = load_run("4-mmlu", "default")
    s = sample(recorded, models=['mistral-nemo', 'mistral-small'], conditions=['no_belief', 'stored_incorrect'], n_items=10)
    print(f"{len(s)} trials. The same trials cost us about US${estimate_cost(s):.4f}.")
    client = Client(get_key(), cap_usd=CAP_USD)
    live = rerun_mcq(client, s, "4-mmlu", "default", load_items("mmlu"))
    display(compare(s, live, "false"))
else:
    print("Replay mode. Set MODE = 'live' above to rerun a small sample with your own key.")